# Deeper investigation — Investigating joins

**Worked solution** · [All exercises](../../index.html) · [Setup](../../README.md)

## What you’ll learn

- Use semi and anti joins to investigate which sales have a product match.
- Trace how duplicate lookup keys affect the number of joined rows and the sales total.

Optional. Complete [Exercise 4](../04-join-aggregate.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](../00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../../RECOVERY.md).

In [1]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 19:33:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="extension-joins"></a>
## Your task

Use an anti join to find accepted sales without a product, and a semi join to find those with one. Call them `unmatched` and `matched`.

Then duplicate the B1 lookup row in a separate `duplicate_products` DataFrame. Make an unchecked left join called `multiplied`. Inspect its count and amount sum. Why does the lookup validation matter? Do not replace `products`.

In [2]:
unmatched = accepted.join(products, on="product_id", how="left_anti")
matched = accepted.join(products, on="product_id", how="left_semi")
duplicate_products = products.unionByName(products.filter(F.col("product_id") == "B1"))
multiplied = accepted.join(duplicate_products, on="product_id", how="left")
unmatched.select("sale_id", "product_id").show()
multiplied.agg(F.count("*").alias("sales"), F.sum("amount").alias("total")).show()

+-------+----------+
|sale_id|product_id|
+-------+----------+
|     s4|        M1|
+-------+----------+



+-----+------+
|sales| total|
+-----+------+
|    8|150.00|
+-----+------+



In [3]:
check.join_experiment(unmatched, matched, multiplied)
check.lookup(products)

Duplicate lookup demonstrated: eight joined rows, total 150.00.


Lookup check passed: one row per product key.


<details><summary>Hint</summary>

Use `left_anti` and `left_semi` to test match existence. `unionByName` can deliberately add a duplicate. The extra matches multiply rows before aggregation.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [4]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Session stopped; exercise files are under runs/run-1633ef004f


Return to [all exercises](../../index.html).